In [1]:
# ====================================================================
# EXPERIMENT 1: CFG Rule Validation & Parse Tree Construction
# ====================================================================

class CFGParser:
    def __init__(self):
        # Lexical category definitions
        self.lexicon = {
            'Det': {'the', 'a'},
            'N': {'dog', 'cat', 'bone'},
            'V': {'chased', 'found'}
        }

    def parse_cfg(self, sentence: str):
        # 1. Normalize and tokenize
        tokens = sentence.strip().lower().split()

        # 2. Check rule length constraint: S -> NP(2) VP(V(1) NP(2)) = 5 tokens
        if len(tokens) != 5:
            return "Invalid Sentence"

        det1, n1, v, det2, n2 = tokens

        # 3. Validate lexical category memberships
        if (det1 in self.lexicon['Det'] and
            n1 in self.lexicon['N'] and
            v in self.lexicon['V'] and
            det2 in self.lexicon['Det'] and
            n2 in self.lexicon['N']):

            # 4. Construct tree dictionary
            tree = {
                'root': 'S',
                'children': [
                    {
                        'node': 'NP',
                        'children': [
                            {'node': f'Det → {det1}'},
                            {'node': f'N → {n1}'}
                        ]
                    },
                    {
                        'node': 'VP',
                        'children': [
                            {'node': f'V → {v}'},
                            {
                                'node': 'NP',
                                'children': [
                                    {'node': f'Det → {det2}'},
                                    {'node': f'N → {n2}'}
                                ]
                            }
                        ]
                    }
                ]
            }
            return self._format_tree(tree)
        else:
            return "Invalid Sentence"

    def _format_tree(self, tree: dict) -> str:
        lines = ["S"]
        np1 = tree['children'][0]
        lines.append("├── NP")
        lines.append(f"│   ├── {np1['children'][0]['node']}")
        lines.append(f"│   └── {np1['children'][1]['node']}")

        vp = tree['children'][1]
        lines.append("└── VP")
        lines.append(f"    ├── {vp['children'][0]['node']}")
        lines.append("    └── NP")
        lines.append(f"        ├── {vp['children'][1]['children'][0]['node']}")
        lines.append(f"        └── {vp['children'][1]['children'][1]['node']}")
        return "\n".join(lines)


# Standalone function
def parse_cfg(sentence: str):
    parser = CFGParser()
    return parser.parse_cfg(sentence)


# Run all 5 Test Cases
exp1_test_cases = [
    "the dog chased a cat",
    "a cat found the bone",
    "dog chased cat",
    "the bone found a dog",
    "chased the dog bone"
]

print("=" * 60)
print("EXPERIMENT 1: CFG PARSE TREE TEST RESULTS")
print("=" * 60)
for sent in exp1_test_cases:
    print(f"\nInput: \"{sent}\"")
    res = parse_cfg(sent)
    if res == "Invalid Sentence":
        print(f"Output: {res}")
    else:
        print("Output:\n" + res)

EXPERIMENT 1: CFG PARSE TREE TEST RESULTS

Input: "the dog chased a cat"
Output:
S
├── NP
│   ├── Det → the
│   └── N → dog
└── VP
    ├── V → chased
    └── NP
        ├── Det → a
        └── N → cat

Input: "a cat found the bone"
Output:
S
├── NP
│   ├── Det → a
│   └── N → cat
└── VP
    ├── V → found
    └── NP
        ├── Det → the
        └── N → bone

Input: "dog chased cat"
Output: Invalid Sentence

Input: "the bone found a dog"
Output:
S
├── NP
│   ├── Det → the
│   └── N → bone
└── VP
    ├── V → found
    └── NP
        ├── Det → a
        └── N → dog

Input: "chased the dog bone"
Output: Invalid Sentence


In [2]:
# ====================================================================
# EXPERIMENT 2: Agreement and Feature Structure Checking
# ====================================================================

# Feature Dictionaries (Attribute-Value Matrices)
SUBJECT_FEATURES = {
    'he':  {'category': 'NP', 'NUM': 'singular', 'PERS': '3rd'},
    'she': {'category': 'NP', 'NUM': 'singular', 'PERS': '3rd'},
    'it':  {'category': 'NP', 'NUM': 'singular', 'PERS': '3rd'},
    'we':  {'category': 'NP', 'NUM': 'plural',   'PERS': '1st'}
}

VERB_FEATURES = {
    'eats':   {'category': 'V', 'NUM': 'singular', 'PERS': '3rd'},
    'sleeps': {'category': 'V', 'NUM': 'singular', 'PERS': '3rd'},
    'eat':    {'category': 'V', 'NUM': 'plural',   'PERS': 'non-3rd'},
    'sleep':  {'category': 'V', 'NUM': 'plural',   'PERS': 'non-3rd'}
}

def check_agreement(subject: str, verb: str) -> bool:
    subj = subject.strip().lower()
    vrb = verb.strip().lower()

    # Check vocabulary coverage
    if subj not in SUBJECT_FEATURES or vrb not in VERB_FEATURES:
        return False

    subj_feats = SUBJECT_FEATURES[subj]
    verb_feats = VERB_FEATURES[vrb]

    # Subject-verb number agreement constraint
    return subj_feats['NUM'] == verb_feats['NUM']


# Run all 5 Test Cases
exp2_test_cases = [
    ("he", "eats"),
    ("he", "eat"),
    ("we", "eat"),
    ("we", "sleeps"),
    ("she", "sleeps")
]

print("=" * 60)
print("EXPERIMENT 2: SUBJECT-VERB AGREEMENT TEST RESULTS")
print("=" * 60)
for subj, vrb in exp2_test_cases:
    result = check_agreement(subj, vrb)
    s_feat = SUBJECT_FEATURES[subj]
    v_feat = VERB_FEATURES[vrb]
    print(f"'{subj}' ({s_feat['NUM']}) + '{vrb}' ({v_feat['NUM']}) -> {result}")

EXPERIMENT 2: SUBJECT-VERB AGREEMENT TEST RESULTS
'he' (singular) + 'eats' (singular) -> True
'he' (singular) + 'eat' (plural) -> False
'we' (plural) + 'eat' (plural) -> True
'we' (plural) + 'sleeps' (singular) -> False
'she' (singular) + 'sleeps' (singular) -> True


In [3]:
# ====================================================================
# EXPERIMENT 3: Probabilistic Context-Free Grammar (PCFG) Parsing
# ====================================================================

class PCFGParser:
    def __init__(self):
        self.p_s = 1.0
        self.p_np = {'tom': 0.7, 'ann': 0.3}
        self.p_vp = {'sings': 0.4, 'dances': 0.6}

    def sentence_probability(self, sentence: str) -> float:
        tokens = sentence.strip().lower().split()

        # Valid sentence must be 2 words: [NP, VP]
        if len(tokens) != 2:
            return 0.0

        w1, w2 = tokens

        # Check vocabulary coverage
        if w1 not in self.p_np or w2 not in self.p_vp:
            return 0.0

        # Compute joint probability: P(S) * P(NP) * P(VP)
        prob = self.p_s * self.p_np[w1] * self.p_vp[w2]
        return round(prob, 4)


# Standalone function
def sentence_probability(sentence: str) -> float:
    parser = PCFGParser()
    return parser.sentence_probability(sentence)


# Run all 5 Test Cases
exp3_test_cases = [
    "Tom sings",
    "Tom dances",
    "Ann sings",
    "Ann dances",
    "Sam sings"
]

print("=" * 60)
print("EXPERIMENT 3: PCFG SENTENCE PROBABILITY TEST RESULTS")
print("=" * 60)
for sent in exp3_test_cases:
    p = sentence_probability(sent)
    print(f"'{sent}' -> Probability: {p:.2f}")

EXPERIMENT 3: PCFG SENTENCE PROBABILITY TEST RESULTS
'Tom sings' -> Probability: 0.28
'Tom dances' -> Probability: 0.42
'Ann sings' -> Probability: 0.12
'Ann dances' -> Probability: 0.18
'Sam sings' -> Probability: 0.00
